<a href="https://colab.research.google.com/github/UPEI-CS/CS1910-Students/blob/main/case_study/Image-Processing/CS1910_Image-Processing_Workbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1: Case Study**
# **Image Processing**

## Save your own copy first

Only the **computer recorder** needs to do this, but everyone may want a copy afterwards. **Do this before you start**, because anything you type into the original is not saved. Choose:

***File->Save a copy in Drive***

or

***Copy to Drive*** if it appears in the above menu.

Google sign-in is required. You will then be able to open the copy from your Google Drive account later (Colab keeps it in the *Colab Notebooks* folder). At the end, share your copy with the rest of your group (**Share** at the top right).

If you can't use Google Drive, choose ***File->Download .ipynb*** and keep the file by email (signing in to a cloud email service) or on a USB key. To open it in another Colab session later, you would have to move it into your Google Drive account.

## How to use this notebook

This is the coding half of the Image Processing case study. You work in a **group of 3–5**: the **paper recorder** fills in the group worksheet (`CS1910_Image-Processing-Partner_Handout.pdf`), the **computer recorder** types in this notebook, and the **speaker** reports to the class. Plan on paper first, then code: each task says which worksheet stage to do before it.

1. **Run the setup cell** (▶️ below) first. It gives you a `check` helper that tells you ✅ or ❌.
2. **Run the cells that load the photos.** Loading takes a few seconds.
3. **Work through the tasks in order.** For each one, answer the *Understand* questions, write your plan in the ✏️ cell (double-click to edit), then code and run the checks. Only try a filter on a photo **after** its checks pass.

Make sure Colab's AI assistance is **off**: this is code your group writes.

## Images as lists

Python has libraries for images, but here an image is just a grid of pixels stored in lists:

- a **grayscale** image is a 2D list of numbers from 0 (black) to 255 (white): `gray[row][col]`;
- a **colour** image is a 3D list: each pixel is `[R, G, B]`, the amounts of red, green and blue, so `img[row][col][0]` is the red part of one pixel.

```python
gray_image = [[0, 0, 0, 255],
              [0, 0, 255, 0],
              [0, 255, 0, 0],
              [255, 0, 0, 0]]          # white on the anti-diagonal

color_image = [[[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]]]   # red + blue = purple
```

Run the next cell to draw them.

In [ ]:
#@title ▶️ Run this cell first
#@markdown This cell sets up two helpers, `check` and `check_case`, that tell you
#@markdown whether your answer is right. You do not need to understand how they work.
import math

_cases_passed = {}


def _same(got, expected):
    if isinstance(expected, float) and isinstance(got, (int, float)) and not isinstance(got, bool):
        return math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-12)
    return type(got) == type(expected) and got == expected


def check(got, expected, label="Answer"):
    if _same(got, expected):
        print(f"✅ {label}: {got!r}")
    elif got == expected:
        print(f"❌ {label}: the value looks right, but it is a {type(got).__name__}"
              f" and should be a {type(expected).__name__}")
    else:
        print(f"❌ {label}: got {got!r}, expected {expected!r}")


def check_case(name, case, got, expected_by_case):
    if case not in expected_by_case:
        print(f"ℹ️ {case!r} is not one of the test cases for {name}. Try one of: "
              + ", ".join(repr(c) for c in expected_by_case))
        return
    passed = _cases_passed.setdefault(name, set())
    check(got, expected_by_case[case], f"{name} with {case!r}")
    if _same(got, expected_by_case[case]):
        passed.add(case)
    still = [repr(c) for c in expected_by_case if c not in passed]
    if still:
        print(f"   {len(passed)} of {len(expected_by_case)} test cases passed. "
              f"Change the input and run again. Still to try: {', '.join(still)}")
    else:
        print(f"   🎉 All {len(expected_by_case)} test cases passed!")


print("✅ Practice checker ready.")

In [ ]:
import matplotlib.pyplot as plt


def show(*images, gray=False):
    """Draw one or more images side by side. Use gray=True for grayscale images."""
    fig = plt.figure(figsize=(6 * len(images), 5))
    for k, image in enumerate(images):
        fig.add_subplot(1, len(images), k + 1)
        plt.axis("off")
        if gray:
            plt.imshow(image, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(image)
    plt.show()


gray_image = [[0, 0, 0, 255],
              [0, 0, 255, 0],
              [0, 255, 0, 0],
              [255, 0, 0, 0]]

color_image = [[[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]],
               [[100, 0, 200], [100, 0, 200], [100, 0, 200]]]

show(gray_image, gray=True)
show(color_image)

## The photos

This cell downloads four photos and turns each one into a 3D list. You do not need to understand `load_jpg`: it uses libraries (`requests`, `PIL`, `numpy`) to read a JPG file.

In [ ]:
import io
import requests
import numpy as np
from PIL import Image


def load_jpg(url):
    """Download a JPG image and return it as a 3D list of [R, G, B] pixels."""
    response = requests.get(url)
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content))
    return np.asarray(img).tolist()


IMAGES = "https://raw.githubusercontent.com/UPEI-CS/CS1910-Students/refs/heads/main/case_study/Image-Processing/images"
im_green_rock = load_jpg(IMAGES + "/green_rock.jpg")
im_light_house = load_jpg(IMAGES + "/lighthouse_top.jpg")
im_panmure = load_jpg(IMAGES + "/panmure_beach.jpg")
im_sunrise = load_jpg(IMAGES + "/sunrise.jpg")

print(f"green rock: height {len(im_green_rock)}, width {len(im_green_rock[0])}")
show(im_green_rock, im_light_house)
show(im_panmure, im_sunrise)

## Task 1: Images as lists (worksheet Stages 0–1)

Do Stage 1 of the worksheet, then write `size(img)`, which returns the image's `(height, width)` as a tuple.

**Understand:**
- Which `len` gives the number of rows? Which gives the number of pixels in a row?
- Does your function work for a grayscale (2D) image as well as a colour (3D) one?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 

In [ ]:
def size(img):
    """Return (height, width) of an image."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
img = [[[255, 0, 0], [0, 255, 0]], [[0, 0, 255], [100, 0, 200]]]
check(size(img), (2, 2), "the worksheet's 2 x 2 image")
check(size(color_image), (4, 3), "color_image")
check(size(gray_image), (4, 4), "gray_image (2D)")
check(img[1][0][2], 255, "img[1][0][2]")

## Task 2: Pixel filters (worksheet Stage 2)

Write three functions. Each one builds and returns a **new** image; it must not change `img_list`.

1. `color_invert(img_list)`: each part becomes `255 - value`. Purple `[100, 0, 200]` becomes `[155, 255, 55]`.
2. `warm_filter(img_list)`: red goes up by 25 (but not above 255) and blue goes down by 25 (but not below 0). Green is unchanged.
3. `grayscale(img_list)`: each pixel becomes **one number**, `round(0.34 * R + 0.5 * G + 0.16 * B)`, so the result is a **2D** list.

**Understand:**
- How do you build a new image row by row? (Start with an empty list, and append a new row for each row.)
- What do `min(value, 255)` and `max(value, 0)` do?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def color_invert(img_list):
    """Return a new image with every colour part replaced by 255 minus it."""
    height, width = size(img_list)
    inverted = []
    for row in range(height):
        new_row = []
        for col in range(width):
            r, g, b = img_list[row][col]
            # Your code here: append the inverted pixel to new_row

        inverted.append(new_row)
    return inverted


def warm_filter(img_list):
    """Return a new image with red raised by 25 (max 255) and blue lowered by 25 (min 0)."""
    # Your code here
    pass


def grayscale(img_list):
    """Return a new 2D image of intensities: round(0.34 R + 0.5 G + 0.16 B)."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
pixels = [[[100, 0, 200], [240, 10, 20], [255, 255, 255]]]   # the worksheet's three pixels
check(color_invert(pixels), [[[155, 255, 55], [15, 245, 235], [0, 0, 0]]], "color_invert")
check(warm_filter(pixels), [[[125, 0, 175], [255, 10, 0], [255, 255, 230]]], "warm_filter")
check(grayscale(pixels), [[66, 90, 255]], "grayscale (a 2D list)")
check(pixels, [[[100, 0, 200], [240, 10, 20], [255, 255, 255]]], "the original image is unchanged")

### Try your filters on a photo

When the checks pass, choose a photo and a filter and run the cell. For `grayscale`, set `gray=True`.

In [ ]:
# Choose one of: im_green_rock, im_light_house, im_panmure, im_sunrise
input_img = im_panmure
# Choose one of: color_invert, warm_filter, grayscale
fn = warm_filter

show(input_img, fn(input_img), gray=False)

## Task 3: Neighbourhood filters (worksheet Stage 3)

Write `average(img_list, sz)` and `median(img_list, sz)`. For each pixel, look at the block of pixels up to `sz` away in every direction (a `3 x 3` block when `sz` is 1). For each colour part separately:

- `average`: the mean of the block, rounded with `round`;
- `median`: sort the block's values and take the middle one, `values[len(values) // 2]`.

A pixel too close to the edge to have a full block is **copied unchanged** into the new image.

**Understand:**
- Which rows does the block for pixel `(row, col)` cover? Write the `range`. Is the last row included?
- How many pixels are in a block when `sz` is 2?
- Your worksheet's 5 x 5 grid is in the checks below (as a gray colour image). What should the centre become?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

In [ ]:
def average(img_list, sz):
    """Return a new image where each pixel is the rounded mean of its (2*sz+1) x (2*sz+1) block."""
    # Your code here
    pass


def median(img_list, sz):
    """Return a new image where each pixel is the median of its (2*sz+1) x (2*sz+1) block."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
def one_part(img):
    """The red part of each pixel, as a 2D list (handy for reading results)."""
    return [[pixel[0] for pixel in row] for row in img]

dot = [[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
       [[0, 0, 0], [255, 255, 255], [0, 0, 0]],
       [[0, 0, 0], [0, 0, 0], [0, 0, 0]]]
grid = [[10, 10, 10, 10, 10], [10, 20, 30, 20, 10], [10, 30, 200, 30, 10], [10, 20, 30, 20, 10], [10, 10, 10, 10, 10]]
grid_img = [[[v, v, v] for v in row] for row in grid]

check(average(dot, 1)[1][1], [28, 28, 28], "average of the white dot")
check(median(dot, 1)[1][1], [0, 0, 0], "median of the white dot")
check(one_part(average(grid_img, 1)), [[10, 10, 10, 10, 10], [10, 37, 40, 37, 10], [10, 40, 44, 40, 10],
                                       [10, 37, 40, 37, 10], [10, 10, 10, 10, 10]], "average of the worksheet grid")
check(one_part(median(grid_img, 1)), [[10, 10, 10, 10, 10], [10, 10, 20, 10, 10], [10, 20, 30, 20, 10],
                                      [10, 10, 20, 10, 10], [10, 10, 10, 10, 10]], "median of the worksheet grid")
check(one_part(average(grid_img, 2))[2][2], 22, "average with sz = 2 (only the centre changes)")
check(dot[1][1], [255, 255, 255], "the original image is unchanged")

### Try them on a photo

Each pass visits about 750,000 pixels, so use `sz` of 1 or 2: larger blocks are much slower.

In [ ]:
input_img = im_light_house
fn = median          # average or median
sz = 2

show(input_img, fn(input_img, sz))

## Task 4: Sharpen (worksheet Stage 4)

Write `sharpen(orig, smooth)`. For each colour part of each pixel, `diff = orig - smooth` and the new value is `orig + diff`, kept between 0 and 255.

**Understand:**
- Why does `orig - smooth` contain only the sharp detail?
- What does `min(255, max(0, value))` do to 356? To -12?

✏️ **Your group's plan** (double-click to edit; write it before you code):

1. 
2. 

In [ ]:
def sharpen(orig, smooth):
    """Return orig + (orig - smooth), with every part kept between 0 and 255."""
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(one_part(sharpen(grid_img, average(grid_img, 1))),
      [[10, 10, 10, 10, 10], [10, 3, 20, 3, 10], [10, 20, 255, 20, 10], [10, 3, 20, 3, 10], [10, 10, 10, 10, 10]],
      "sharpen the worksheet grid")
check(sharpen([[[100, 0, 200]]], [[[100, 0, 200]]]), [[[100, 0, 200]]], "nothing to sharpen")

### Try it on a photo

In [ ]:
original = im_green_rock
smoothed = average(original, 2)
show(original, sharpen(original, smoothed))

## Before you leave

Tick off each statement your group can honestly agree with. For any you can't, go back to the matching task or worksheet stage, and bring your questions to the next class.

- [ ] I can index into a 2D and a 3D list, and find an image's height and width.
- [ ] I can use nested loops to visit every pixel, and build a new image instead of changing the original.
- [ ] I can explain the difference between an average and a median filter.
- [ ] I can test a function on a tiny example I can check by hand.

**Share your copy** of this notebook with the rest of your group before you go.

### On your own

Edge pixels are copied unchanged here. Other ways to handle them: pad the image with zeros, repeat the border pixels, or wrap around to the other side. Try one in your own copy.